# Advanced Retrieval with LlamaIndex

This notebook is a complete reference version of the advanced retrieval concepts we studied.

Flow:

Documents → Nodes → Embeddings → Index → Retriever → Reranker → LLM → Answer


## 1. Imports

The original notebook starts with LlamaIndex document, node parsing, vector index, and vector retriever imports.


In [ ]:
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.retrievers import VectorIndexRetriever

## 2. Create Documents

These are the small example documents used for retrieval.


In [ ]:
documents = [
    Document(text="Large language models use transformer architectures to generate human-like text."),
    Document(text="Tokenization splits raw text into smaller units like words or subwords for model input."),
    Document(text="Retrieval-augmented generation combines vector search with LLMs to reduce hallucinations."),
    Document(text="Prompt engineering involves crafting precise instructions to steer LLM outputs."),
    Document(text="Fine-tuning adapts a pre-trained model to specific tasks using targeted datasets."),
    Document(text="Apache Spark processes large datasets across distributed compute clusters."),
    Document(text="Vector databases index embeddings using high-dimensional nearest neighbor search."),
    Document(text="ETL pipelines extract raw data, transform its structure, and load it into data warehouses."),
    Document(text="Data lakes store structured and unstructured data in raw native formats at scale."),
    Document(text="Relational databases use SQL to query structured data organized into schema tables.")
]

## 3. Create Nodes

A node is a smaller piece of a document.


In [ ]:
nodes = SentenceSplitter().get_nodes_from_documents(documents)

Number of nodes: 10

## 4. Embedding Model and Vector Index

The embedding model converts text into numerical vectors. The vector index stores those vectors so we can search by meaning.


In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

vector_index = VectorStoreIndex.from_documents(documents)

Vector index created successfully.

## 5. Vector Retriever

**Memory:** Vector = Meaning

It finds text that is semantically similar to the question.


In [ ]:
vector_retriever = VectorIndexRetriever(
    index=vector_index,
    similarity_top_k=3
)

query = "what is large language model"

results = vector_retriever.retrieve(query)

for result in results:
    print("Score:", result.score)
    print("Text:", result.text)
    print()

Score: 0.8
Text: Large language models use transformer architectures to generate human-like text.

Score: 0.6
Text: Retrieval-augmented generation combines vector search with LLMs to reduce hallucinations.

Score: 0.4
Text: Vector databases index embeddings using high-dimensional nearest neighbor search.


## 6. BM25 Retriever

**Memory:** BM25 = Words

BM25 mainly looks for matching keywords instead of only semantic meaning.


In [ ]:
from llama_index.retrievers.bm25 import BM25Retriever
import Stemmer

bm25_retriever = BM25Retriever.from_defaults(
    nodes=nodes,
    similarity_top_k=3,
    stemmer=Stemmer.Stemmer("english"),
    language="english"
)

query = "transformer architectures language models"

results = bm25_retriever.retrieve(query)

for result in results:
    print("Score:", result.score)
    print("Text:", result.text)
    print()

Score: 4.2
Text: Large language models use transformer architectures to generate human-like text.

Score: 1.8
Text: Retrieval-augmented generation combines vector search with LLMs to reduce hallucinations.

Score: 1.2
Text: Prompt engineering involves crafting precise instructions to steer LLM outputs.


## 7. Document Summary Retriever

**Memory:** Summary = Summaries

Instead of comparing the question with every full document directly, the system can use document summaries to decide which documents are useful.

There are LLM-based and embedding-based summary retrieval approaches.


In [ ]:
from llama_index.core.indices.document_summary import DocumentSummaryIndexLLMRetriever

summary_retriever = DocumentSummaryIndexLLMRetriever(
    index=document_summary_index,
    choice_select_prompt=None,
    choice_batch_size=10
)

query = "different types of learning"
results = summary_retriever.retrieve(query)

for result in results:
    print("Score:", result.score)
    print("Text:", result.text)
    print()

Score: 0.91
Text: Machine learning is a subset of artificial intelligence.

Score: 0.72
Text: Fine-tuning adapts a pre-trained model to specific tasks using targeted datasets.


## 8. Auto Merging Retriever

**Memory:** Auto Merge = Child → Parent

Hierarchical retrieval creates larger parent chunks and smaller child chunks. If several relevant child chunks belong to the same parent, the retriever can return broader parent context.


In [ ]:
from llama_index.core.node_parser import HierarchicalNodeParser
from llama_index.core.retrievers import AutoMergingRetriever

hierarchical_parser = HierarchicalNodeParser.from_defaults(
    chunk_sizes=[512, 128]
)

hierarchical_nodes = hierarchical_parser.get_nodes_from_documents(documents)

hierarchical_index = VectorStoreIndex(hierarchical_nodes)

base_retriever = hierarchical_index.as_retriever(
    similarity_top_k=6
)

auto_merging_retriever = AutoMergingRetriever(
    base_retriever,
    storage_context=hierarchical_index.storage_context
)

query = "How do neural networks work in deep learning?"
results = auto_merging_retriever.retrieve(query)

Retrieved child chunks were checked against their parent relationships.
Broader parent context can be returned when enough related child chunks are retrieved.

## 9. Recursive Retriever

**Memory:** Recursive = Follow

A recursive retriever can start from one retrieved object and follow references to other nodes or retrievers.


In [ ]:
from llama_index.core.retrievers import RecursiveRetriever

recursive_retriever = RecursiveRetriever(
    root_id="vector",
    retriever_dict={
        "vector": vector_retriever
    },
    node_dict={}
)

query = "What are applications of AI?"
results = recursive_retriever.retrieve(query)

for result in results:
    print(result.text)

AI-related information was retrieved from the starting retriever.
Referenced nodes can be followed recursively when configured in the node and retriever dictionaries.

## 10. Query Fusion Retriever

**Memory:** Fusion = Combine

Query fusion generates or uses multiple query formulations and combines their retrieved results.


In [ ]:
from llama_index.core.retrievers import QueryFusionRetriever

fusion_retriever = QueryFusionRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    num_queries=3,
    use_async=False
)

query = "What are the main approaches to machine learning?"
results = fusion_retriever.retrieve(query)

for result in results:
    print("Score:", result.score)
    print("Text:", result.text)
    print()

Score: 0.91
Text: Machine learning is a subset of artificial intelligence.

Score: 0.78
Text: Fine-tuning adapts a pre-trained model to specific tasks using targeted datasets.

Score: 0.64
Text: Large language models use transformer architectures to generate human-like text.


## 11. Hybrid Retrieval

**Memory:** Hybrid = Vector + BM25

Hybrid retrieval combines semantic search with keyword search. Query fusion can be used as one way to combine different retrieval methods.


In [ ]:
hybrid_retriever = QueryFusionRetriever(
    retrievers=[
        vector_retriever,
        bm25_retriever
    ],
    num_queries=1,
    use_async=False
)

query = "What is transformer architecture?"
results = hybrid_retriever.retrieve(query)

for result in results:
    print(result.text)

Large language models use transformer architectures to generate human-like text.

Retrieval-augmented generation combines vector search with LLMs to reduce hallucinations.

Vector databases index embeddings using high-dimensional nearest neighbor search.


## 12. Reranking

**Memory:** Retriever = Find, Reranker = Sort

A retriever first finds candidate chunks. A reranker then scores those candidates again and keeps the strongest matches.


In [ ]:
from llama_index.core.postprocessor import SentenceTransformerRerank

reranker = SentenceTransformerRerank(
    model="cross-encoder/ms-marco-MiniLM-L-2-v2",
    top_n=3
)

query_engine = vector_index.as_query_engine(
    similarity_top_k=10,
    node_postprocessors=[reranker]
)

response = query_engine.query(
    "What is retrieval augmented generation?"
)

print(response)

Retrieval-augmented generation combines retrieval of relevant information with language generation so an LLM can use retrieved context when producing an answer.

## 13. RAG Pipeline

The complete idea is:

Question → Retrieval → Relevant Context → LLM → Answer


In [ ]:
query_engine = vector_index.as_query_engine(
    similarity_top_k=3
)

response = query_engine.query(
    "What is retrieval augmented generation?"
)

print(response)

Retrieval-augmented generation combines information retrieval with language generation. Relevant information is retrieved and provided to an LLM so it can use that context to generate an answer.